# Attention Head Heatmap Explorer

Demonstrates the small-multiples heatmap and aggregation visualizations
for comparing all attention heads in a single layer at once.

**Key ideas vs arc diagrams:**
- Heatmaps show the *full* N×N attention matrix — not just top-k edges.
- Small multiples let you scan all heads simultaneously at the same color scale.
- The aggregated std-dev view highlights which positions differ across heads
  (i.e., *specialised* heads) vs. positions every head agrees on.
- Per-head metrics (entropy, diagonal concentration, mean distance) let you
  rank / filter heads before generating any plot.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from attention_heatmaps import load_attention_matrix, plot_head_heatmaps, plot_aggregated_heatmap
from attention_heatmaps import compute_head_metrics
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd

In [ ]:
# ---- Configuration ----
# Point these at your actual attention output directory.
ATTN_DIR  = "../outputs/attention_images_6KWC_demo_tri_18"  # adjust as needed
PROTEIN   = "6KWC"
LAYER_IDX = 47

# MSA row attention file
MSA_FILE  = os.path.join(ATTN_DIR, f"msa_row_attn_layer{LAYER_IDX}.txt")
# Triangle start attention file (residue 18 in the demo)
TRI_FILE  = os.path.join(ATTN_DIR, f"triangle_start_attn_layer{LAYER_IDX}_residue_idx_18.txt")

print("MSA file exists: ", os.path.exists(MSA_FILE))
print("Tri file exists: ", os.path.exists(TRI_FILE))

## 1. Load attention matrices

In [ ]:
msa_matrices, seq_len = load_attention_matrix(MSA_FILE)
print(f"Loaded {len(msa_matrices)} heads, sequence length = {seq_len}")
print(f"Head indices: {sorted(msa_matrices.keys())}")

## 2. Per-head diagnostic metrics

Rank heads *before* plotting — useful when you have dozens of heads and
want to focus on interesting ones first.

In [ ]:
metrics = compute_head_metrics(msa_matrices)
df = pd.DataFrame(metrics).T
df.index.name = "head"
df = df.sort_values("entropy", ascending=False)
df

**Reading the table:**
- **entropy ↑** → attention is spread across many residues (diffuse head)
- **sparsity ↑** → most pairs ignored; only a few strong connections
- **diagonal_concentration ↑** → head focuses on nearby/sequential residues
- **mean_attention_distance ↑** → head is attending long-range
- **max_weight** → absolute strength of the strongest single connection

## 3. Small-multiples heatmap — all heads at once

In [ ]:
grid_path = "/tmp/msa_row_grid.png"
plot_head_heatmaps(
    msa_matrices,
    output_path=grid_path,
    protein=PROTEIN,
    layer_idx=LAYER_IDX,
    attention_type="msa_row",
    normalization="global_max",
    threshold_pct=0,
    cols=4,
    cmap="viridis",
    dpi=120,
)

img = mpimg.imread(grid_path)
plt.figure(figsize=(14, 8))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

### 3a. With 50th-percentile threshold (cut noise)

In [ ]:
thresh_path = "/tmp/msa_row_grid_thresh50.png"
plot_head_heatmaps(
    msa_matrices,
    output_path=thresh_path,
    protein=PROTEIN,
    layer_idx=LAYER_IDX,
    attention_type="msa_row",
    normalization="global_max",
    threshold_pct=50,
    cols=4,
    cmap="viridis",
    dpi=120,
)

img = mpimg.imread(thresh_path)
plt.figure(figsize=(14, 8))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

## 4. Aggregated heatmaps

Each aggregation asks a different question:
- **mean**: where does the layer collectively attend?
- **max**: which pairs does *any* head find important?
- **std**: which pairs are *contested* — high variation across heads signals specialisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, agg in zip(axes, ["mean", "max", "std"]):
    tmp = f"/tmp/msa_agg_{agg}.png"
    plot_aggregated_heatmap(
        msa_matrices,
        output_path=tmp,
        protein=PROTEIN,
        layer_idx=LAYER_IDX,
        attention_type="msa_row",
        agg=agg,
        cmap="magma",
        dpi=100,
    )
    ax.imshow(mpimg.imread(tmp))
    ax.axis('off')
    ax.set_title(f"Aggregation: {agg}", fontsize=11)

plt.tight_layout()
plt.show()

## 5. Triangle start attention with residue highlight

In [ ]:
if os.path.exists(TRI_FILE):
    tri_matrices, tri_seq_len = load_attention_matrix(TRI_FILE)
    tri_grid_path = "/tmp/tri_start_grid.png"
    plot_head_heatmaps(
        tri_matrices,
        output_path=tri_grid_path,
        protein=PROTEIN,
        layer_idx=LAYER_IDX,
        attention_type="triangle_start",
        normalization="global_max",
        threshold_pct=0,
        cols=4,
        highlight_residue=18,
        cmap="plasma",
        dpi=120,
    )
    img = mpimg.imread(tri_grid_path)
    plt.figure(figsize=(14, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print("Triangle start file not found — skipping.")

## 6. Interactive web app

For interactive exploration (filter heads, toggle normalization, sort by
metric, download PNGs), run the Flask server:

```bash
cd attention_heatmaps/web
python app.py \
    --attn_dir /path/to/attn_maps \
    --protein 6KWC \
    --default_layer 47
```

Then open **http://localhost:5050**.